In [ ]:
import pandas as pd
import numpy as np

def run_data_cleaning_pipeline():
    print("==================================================")
    print("KENYAN RETAILER CAPSTONE: DATA CLEANING PIPELINE")
    print("==================================================\n")

    # 1. Load Raw Datasets
    orders = pd.read_csv('orders.csv')
    order_items = pd.read_csv('order_items.csv')
    customers = pd.read_csv('customers.csv')
    payments = pd.read_csv('payments.csv')
    products = pd.read_csv('products.csv')

    print("--- RAW DATA COUNTS ---")
    print(f"Orders rows: {len(orders)}, Order Items rows: {len(order_items)}")
    print(f"Customers rows: {len(customers)}, Payments rows: {len(payments)}, Products rows: {len(products)}\n")

    # ==========================================
    # STEP 1: FIX DATA TYPES & PARSE DATES
    # ==========================================
    print("Executing Step 1: Fixing data types and standardizing dates...")
    
    # Track before state
    orders_null_dates_before = orders['order_date'].isna().sum()
    
    # Convert dates
    orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
    orders['delivered_at'] = pd.to_datetime(orders['delivered_at'], errors='coerce')
    customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')
    payments['paid_at'] = pd.to_datetime(payments['paid_at'], errors='coerce')
    
    print(f" -> Converted datetime columns across orders, customers, and payments.")

    # ==========================================
    # STEP 2: HANDLE MISSING VALUES
    # ==========================================
    print("\nExecuting Step 2: Imputing / handling missing values...")
    
    # Missing delivery fees in orders
    missing_fees_before = orders['delivery_fee'].isna().sum()
    orders['delivery_fee'] = orders['delivery_fee'].fillna(0.0)
    missing_fees_after = orders['delivery_fee'].isna().sum()
    
    # Missing discount percentages in order items
    missing_disc_before = order_items['discount_pct'].isna().sum()
    order_items['discount_pct'] = order_items['discount_pct'].fillna(0.0)
    missing_disc_after = order_items['discount_pct'].isna().sum()

    # Missing loyalty tiers in customers
    missing_tier_before = customers['loyalty_tier'].isna().sum()
    customers['loyalty_tier'] = customers['loyalty_tier'].fillna('Bronze').str.strip().str.title()
    missing_tier_after = customers['loyalty_tier'].isna().sum()

    print(f" -> Delivery fees fixed (Missing before: {missing_fees_before} | After: {missing_fees_after})")
    print(f" -> Order discounts fixed (Missing before: {missing_disc_before} | After: {missing_disc_after})")
    print(f" -> Customer loyalty tiers imputed (Missing before: {missing_tier_before} | After: {missing_tier_after})")

    # ==========================================
    # STEP 3: REMOVE DUPLICATES
    # ==========================================
    print("\nExecuting Step 3: Checking and removing duplicates...")
    
    orders_dup_before = orders.duplicated().sum()
    orders = orders.drop_duplicates()
    orders_dup_after = orders.duplicated().sum()

    order_items_dup_before = order_items.duplicated().sum()
    order_items = order_items.drop_duplicates()
    order_items_dup_after = order_items.duplicated().sum()

    print(f" -> Orders duplicates removed (Before: {orders_dup_before} | After: {orders_dup_after})")
    print(f" -> Order Items duplicates removed (Before: {order_items_dup_before} | After: {order_items_dup_after})")

    # ==========================================
    # STEP 4: HANDLE OUTLIERS & STANDARDIZE TEXT
    # ==========================================
    print("\nExecuting Step 4: Text standardization & outlier handling...")
    
    # Standardize payment methods
    payments['payment_method'] = (
        payments['payment_method']
        .str.strip()
        .str.title()
        .replace({'M-Pesa': 'M-Pesa', 'Card': 'Card', 'Cash': 'Cash', 'Refund': 'Refund'})
    )
    
    # Cap extreme outlier quantities if any exist (e.g. quantity > 100)
    extreme_qty_before = (order_items['quantity'] > 100).sum()
    # For analytical integrity, we retain valid bulk purchases but flag/inspect
    print(f" -> Standardized payment methods (M-Pesa, Cash, Card).")
    print(f" -> Inspected quantity outliers (Items with qty > 100: {extreme_qty_before})")

    # ==========================================
    # STEP 5: FEATURE ENGINEERING
    # ==========================================
    print("\nExecuting Step 5: Engineering analytical features...")
    
    # Calculate Net Revenue per order item
    order_items['net_revenue'] = (
        order_items['quantity'] * order_items['unit_price'] * (1 - order_items['discount_pct'] / 100.0)
    )
    
    # Merge order items with orders for temporal features
    print(f" -> Engineered net line-item revenue feature (Total items calculated: {len(order_items)})")

    # Save cleaned outputs for downstream SQL and Modeling milestones
    orders.to_csv('cleaned_orders.csv', index=False)
    order_items.to_csv('cleaned_order_items.csv', index=False)
    customers.to_csv('cleaned_customers.csv', index=False)
    payments.to_csv('cleaned_payments.csv', index=False)
    
    print("\n================================================")
    print("PIPELINE EXECUTION COMPLETE: ALL FILES CLEANED")
    print("================================================")

if __name__ == '__main__':
    run_data_cleaning_pipeline()